# Assignment 04 — RAG Basics
## Agriculture Knowledge Assistant

### Objective
Build a Retrieval-Augmented Generation (RAG) pipeline over agriculture-related PDF documents. The system extracts document text, creates overlapping chunks, generates local embeddings, stores them in ChromaDB, retrieves relevant passages, and uses Gemini to answer questions grounded in the retrieved context.

### Workflow
1. Collect and inspect agriculture PDF documents.
2. Extract text, using OCR for scanned PDFs when necessary.
3. Split text into 500-character chunks with 50-character overlap.
4. Generate 384-dimensional embeddings with `all-MiniLM-L6-v2`.
5. Store chunks, embeddings, and source metadata in ChromaDB.
6. Retrieve the top 5 relevant chunks for a query.
7. Generate a grounded answer with source references.
8. Test refusal for questions not answered by the documents.

**Domain:** Agriculture  
**Embedding model:** `sentence-transformers/all-MiniLM-L6-v2`  
**Vector database:** ChromaDB  
**Answer-generation model:** Gemini

## 1. Install and import libraries

In [ ]:
!pip install -q google-genai chromadb pypdf requests beautifulsoup4 sentence-transformers pymupdf pytesseract
!apt-get install -y tesseract-ocr

In [33]:
!pip install -q sentence-transformers

In [ ]:
import os
import io
import numpy as np
import chromadb
import pypdf
import pymupdf
import pytesseract
from PIL import Image
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from google.colab import userdata
from google import genai

print("All required libraries imported successfully!")

## 2. Upload agriculture PDF documents

In [ ]:
from google.colab import files

uploaded = files.upload()

print(f"\nTotal files uploaded: {len(uploaded)}")
for filename in uploaded:
    print(filename)

Saving BasmatiRiceReviseResige_.pdf to BasmatiRiceReviseResige_ (2).pdf
Saving BulletinNCIPM2010_.pdf to BulletinNCIPM2010_ (2).pdf
Saving cabbageegnlish_.pdf to cabbageegnlish_ (1).pdf
Saving fruits_.pdf to fruits_ (2).pdf
Saving Insect-pests-of-maize-and-their-management.pdf to Insect-pests-of-maize-and-their-management.pdf
Saving maize-diseases-and-their-management.pdf to maize-diseases-and-their-management.pdf
Saving PestofPigeonpea_.pdf to PestofPigeonpea_ (2).pdf
Saving Selected-Agricultural-Technologies-A Compendium.pdf to Selected-Agricultural-Technologies-A Compendium (1).pdf
Saving Swarming_caterpillar_.pdf to Swarming_caterpillar_ (2).pdf
Saving VegetableCropsenglish_.pdf to VegetableCropsenglish_ (1).pdf

Total files uploaded: 10
BasmatiRiceReviseResige_ (2).pdf
BulletinNCIPM2010_ (2).pdf
cabbageegnlish_ (1).pdf
fruits_ (2).pdf
Insect-pests-of-maize-and-their-management.pdf
maize-diseases-and-their-management.pdf
PestofPigeonpea_ (2).pdf
Selected-Agricultural-Technologies-A

## 3. Extract text from PDFs

In [ ]:
import os
from pypdf import PdfReader

documents = []

for filename in uploaded:
    if filename.lower().endswith(".pdf"):
        reader = PdfReader(filename)

        text = ""
        for page in reader.pages:
            text += page.extract_text() or ""

        documents.append({
            "filename": filename,
            "text": text,
            "pages": len(reader.pages)
        })

print("PDF extraction completed!\n")

for doc in documents:
    print(f"File: {doc['filename']}")
    print(f"Pages: {doc['pages']}")
    print(f"Extracted characters: {len(doc['text'])}")
    print("-" * 50)

PDF extraction completed!

File: BasmatiRiceReviseResige_ (2).pdf
Pages: 2
Extracted characters: 7388
--------------------------------------------------
File: BulletinNCIPM2010_ (2).pdf
Pages: 18
Extracted characters: 24273
--------------------------------------------------
File: cabbageegnlish_ (1).pdf
Pages: 2
Extracted characters: 10405
--------------------------------------------------
File: fruits_ (2).pdf
Pages: 78
Extracted characters: 126517
--------------------------------------------------
File: Insect-pests-of-maize-and-their-management.pdf
Pages: 2
Extracted characters: 0
--------------------------------------------------
File: maize-diseases-and-their-management.pdf
Pages: 2
Extracted characters: 0
--------------------------------------------------
File: PestofPigeonpea_ (2).pdf
Pages: 114
Extracted characters: 143253
--------------------------------------------------
File: Selected-Agricultural-Technologies-A Compendium (1).pdf
Pages: 131
Extracted characters: 225532
----

### Inspect extracted text quality

In [ ]:
for doc in documents:
    print("\n" + "=" * 70)
    print("FILE:", doc["filename"])
    print("=" * 70)
    print(doc["text"][:500])


FILE: BasmatiRiceReviseResige_ (2).pdf
Developed by
ICAR-National Research Centre for 
Integrated Pest Management
New Delhi – 110012
INTEGRATED 
PEST MANAGEMENT IN
BASMATI RICE
IPM Strategy with 
Safe Pesticide 
Residue Rice Production 
ETL: 3-5 lesions/leaf early to late tillering or 2-5 neck 2
infected plants/m  at panicle initiation to booting.
Spray tebuconazole 25% WG @750 ml/ha or 
azoxystrobin 18.2% w/w + difenoconazole 11.4% w/w 
SC  @ 0.1%  or 1 ml / litre water or tebuconazole 50% + 
trifloxystrobin 25% WG@200 g/ha: Tricycl

FILE: BulletinNCIPM2010_ (2).pdf
Good Agricultural Practices (GAP) for IPM in Protected Cultivation
 15
Technical Bulletin No. 23
Good Agricultural Practices (GAP)
 for IPM in Protected Cultivation
Naved Sabir, Balraj Singh*, M. Hasan*, R. Sumitha,
Sikha Deka, R.K. Tanwar, D.B. Ahuja , B.S. Tomar**,
O.M. Bambawale & E.M. Khah†
National Centre for Integrated Pest Management (ICAR),
LBS Building, Pusa Campus, New Delhi-110012
*Centre for Protected Cultivat

This prints the first 500 characters from each document.

Why? Character counts alone don't guarantee good extraction. We need to check whether the extracted text is readable and meaningful, especially tables and formatting.

### Inspect a document with potentially noisy extraction

Inspect the questionable document

In [ ]:
doc = next(
    d for d in documents
    if d["filename"].startswith("Selected-Agricultural")
)

print("Document:", doc["filename"])
print("Total characters:", len(doc["text"]))

for position in [0, 5000, 20000, 50000, 100000]:
    print(f"\n--- Position {position} ---")
    print(doc["text"][position:position+500])

Document: Selected-Agricultural-Technologies-A Compendium (1).pdf
Total characters: 225532

--- Position 0 ---
5e le rted
Ag ricu lru rs I T$ch lr$log ies
, ,, Afump*ndilrs,
lndion Council of Agriculturol ReseorchSelec*sd
&gnfi*w**axrm$ Yex$Nmm**6**s
* * *,&ffompe*xd$*pga?$eilec*e*
&gr$cwilturffi [ Yechmotosfies
**.ACom*pes?d$wrxr
ICAR
INDIAN COUNCIL OF AGRICUTTURAL RESEARCH
NEW DETHICompilolion ond Editing
Director (DIPA)
Chief Production Officer
Technicol Officer
MAY 2OO4
Dn. R. Pmsnm
ADG (Asril. Extn.)
A Cnnrnnvnnrr
V K Bnmrr
Asnor Srnsrnr
Published by A Chokrovorty, Director, Directoroie of lnformoii

--- Position 5000 ---
D RICE
Crop % Yield States for which recommended for cultivatlon
duration advantage over
(Days) local check
APHR_I
DRRH_1
APHR_2
MGR-I
CORH_z
ANTRH.I
KRH-I
KRH_2
cl{RH-3
Pant Sankar
Dhan-1
Sahyadri
Narendra
Sankar Dhan-Z
PHB_71
PA 5201 (2ooo)
HRI-120 (2001)
r30-135
125-130
120-125
1 10-1 15
120-125
115-120
120*125
130-135
125-130
115-120
125-r30
125-r30
130-135
1

## 4. OCR scanned documents

Some PDFs may contain scanned page images instead of selectable text. The following cells install and verify Tesseract OCR, then extract text from the two scanned maize PDFs.

In [ ]:
import fitz
import pytesseract
from PIL import Image

print("PyMuPDF version:", fitz.__doc__.split()[1])
print("Tesseract version:", pytesseract.get_tesseract_version())
print("OCR libraries ready!")

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


PyMuPDF version: 2.2:
Tesseract version: 5.3.4
OCR libraries ready!


OCR the two maize PDFs

In [ ]:
import pymupdf
import pytesseract
from PIL import Image
import io

maize_files = [
    "Insect-pests-of-maize-and-their-management.pdf",
    "maize-diseases-and-their-management.pdf"
]

ocr_results = {}

for filename in maize_files:
    pdf = pymupdf.open(filename)
    extracted_text = []

    for page_num, page in enumerate(pdf):
        pix = page.get_pixmap(matrix=pymupdf.Matrix(2, 2))
        image = Image.open(io.BytesIO(pix.tobytes("png")))

        text = pytesseract.image_to_string(image, lang="eng")
        extracted_text.append(text)

    full_text = "\n".join(extracted_text)
    ocr_results[filename] = full_text

    print(f"\nFile: {filename}")
    print(f"Pages: {len(pdf)}")
    print(f"Extracted characters: {len(full_text)}")
    print("\nPreview:")
    print(full_text[:700])
    print("-" * 60)


File: Insect-pests-of-maize-and-their-management.pdf
Pages: 2
Extracted characters: 13545

Preview:
Integrated Pest Management Strategies

Sowing to six leaf stage

+ Timely and uniform sowing over larger area.

Ridge and furrow planting method to be followed.

Application of recommended dosage of NPK as basal dose.

Planting of 3-4 rows of napier grass/hybrid napier as trap crops.

Intercropping of maize with legumes, viz., pigeonpea, cowpea, black gram,

kidney bean etc. in2: 1 to 4: 1 ratio.

+ Erection of bird perches @ 10/acre to encourage natural FAW predation by birds.

+ Installation of pheromone traps @ 4/acre soon after sowing and monitor moth
catches.

+ Adoption of clean cultivation to eliminate possible alternate hosts.

* Crushing of egg masses and larvae.

¢ Application of s
------------------------------------------------------------

File: maize-diseases-and-their-management.pdf
Pages: 2
Extracted characters: 10882

Preview:
Ploughing the field deeply during April and

### Add OCR text to the document collection

In [ ]:
for filename, text in ocr_results.items():
    for doc in documents:
        if doc["filename"].startswith(filename):
            doc["text"] = text
            doc["extraction_method"] = "OCR"
            break

print("Updated document collection:\n")

for doc in documents:
    method = doc.get("extraction_method", "Native PDF")
    print(
        f"{doc['filename']} | "
        f"{len(doc['text'])} characters | "
        f"{method}"
    )

Updated document collection:

BasmatiRiceReviseResige_ (2).pdf | 7388 characters | Native PDF
BulletinNCIPM2010_ (2).pdf | 24273 characters | Native PDF
cabbageegnlish_ (1).pdf | 10405 characters | Native PDF
fruits_ (2).pdf | 126517 characters | Native PDF
Insect-pests-of-maize-and-their-management.pdf | 13545 characters | OCR
maize-diseases-and-their-management.pdf | 10882 characters | OCR
PestofPigeonpea_ (2).pdf | 143253 characters | Native PDF
Selected-Agricultural-Technologies-A Compendium (1).pdf | 225532 characters | Native PDF
Swarming_caterpillar_ (2).pdf | 32765 characters | Native PDF
VegetableCropsenglish_ (1).pdf | 67521 characters | Native PDF


## 5. Chunk the extracted documents

Each document is split into fixed-size chunks of approximately **500 characters**, with **50 characters of overlap**. The overlap helps preserve context across chunk boundaries.

In [ ]:
def chunk_text(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0

    while start < len(text):
        chunk = text[start:start + chunk_size]

        if chunk.strip():
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


all_chunks = []

for doc in documents:
    chunks = chunk_text(doc["text"])

    for chunk_id, chunk in enumerate(chunks):
        all_chunks.append({
            "filename": doc["filename"],
            "chunk": chunk_id,
            "text": chunk
        })

print("Chunking completed!")
print("Total documents:", len(documents))
print("Total chunks:", len(all_chunks))

print("\nSample chunk:")
print(all_chunks[0])

Chunking completed!
Total documents: 10
Total chunks: 1478

Sample chunk:
{'filename': 'BasmatiRiceReviseResige_ (2).pdf', 'chunk': 0, 'text': 'Developed by\nICAR-National Research Centre for \nIntegrated Pest Management\nNew Delhi – 110012\nINTEGRATED \nPEST MANAGEMENT IN\nBASMATI RICE\nIPM Strategy with \nSafe Pesticide \nResidue Rice Production \nETL: 3-5 lesions/leaf early to late tillering or 2-5 neck 2\ninfected plants/m  at panicle initiation to booting.\nSpray tebuconazole 25% WG @750 ml/ha or \nazoxystrobin 18.2% w/w + difenoconazole 11.4% w/w \nSC  @ 0.1%  or 1 ml / litre water or tebuconazole 50% + \ntrifloxystrobin 25% WG@200 g/ha: Tricycl'}


**Chunking configuration**

| Parameter | Value |
|---|---:|
| Chunk size | 500 characters |
| Overlap | 50 characters |
| Step size | 450 characters |

chunking configuration is:

Parameter	| Value

Chunk size | 500 characters

Overlap | 50 characters

Step size | 450 characters

Total chunks | 1,478

## 6. Generate local embeddings

The Gemini embedding API free-tier daily quota was reached during development. To complete the assignment without API quota restrictions, this notebook uses the local Sentence Transformers model `all-MiniLM-L6-v2` for **all** chunks and queries. Using one model consistently ensures that document and query vectors share the same embedding space.

In [35]:
!pip install -q sentence-transformers

In [36]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully!


In [37]:
import numpy as np

texts = [chunk["text"] for chunk in all_chunks]

all_embeddings = embedding_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Total embeddings:", len(all_embeddings))
print("Embedding dimensions:", all_embeddings.shape)

Batches:   0%|          | 0/47 [00:00<?, ?it/s]

Total embeddings: 1478
Embedding dimensions: (1478, 384)


### Save the embeddings

Save the embeddings

In [38]:
import numpy as np

np.save("agriculture_embeddings.npy", all_embeddings)

print("Embeddings saved successfully!")

Embeddings saved successfully!


## 7. Create the ChromaDB vector store

ChromaDB stores each chunk together with its embedding and metadata. The persistent client keeps the vector store on disk in the Colab working directory.

In [ ]:
import chromadb

chroma_client = chromadb.PersistentClient(path="./chroma_db")

# Recreate the collection so this notebook can be rerun cleanly.
try:
    chroma_client.delete_collection("agriculture_knowledge")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="agriculture_knowledge",
    metadata={"hnsw:space": "cosine"}
)

print("ChromaDB collection created!")

### Insert chunks, embeddings, and source metadata

In [ ]:
collection.upsert(
    ids=[f"chunk_{i}" for i in range(len(all_chunks))],
    documents=[chunk["text"] for chunk in all_chunks],
    embeddings=all_embeddings.tolist(),
    metadatas=[
        {
            "filename": chunk["filename"],
            "chunk_number": chunk["chunk"]
        }
        for chunk in all_chunks
    ]
)

print("Total records in ChromaDB:", collection.count())

## 8. Retrieve the top 5 relevant chunks

In [42]:
query = "What are the common insect pests affecting maize crops?"

query_embedding = embedding_model.encode(
    [query],
    normalize_embeddings=True
)[0]

results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=5
)

for i in range(5):
    print(f"\nResult {i+1}")
    print("Source:", results["metadatas"][0][i]["filename"])
    print("Chunk:", results["metadatas"][0][i]["chunk_number"])
    print("Distance:", round(results["distances"][0][i], 4))
    print("Text:", results["documents"][0][i][:300])


Result 1
Source: Insect-pests-of-maize-and-their-management.pdf
Chunk: 13
Distance: 0.3082
Text: the
productivity of maize (3.1 t/ha) is much lower than the world average (5.62 t/ha).
Insect pests are one of the major factors responsible for low productivity. In the
context of India the major insect pests of maize which cause economic damage are
spotted stem borer [Chilo partellus (Swinhoe)], p

Result 2
Source: Insect-pests-of-maize-and-their-management.pdf
Chunk: 12
Distance: 0.316
Text: mi Soujanya, S. B. Suby, Jawala Jindal,
J.C. Sekhar, Ph. Romen Sharma and H.S.Jat

——

gor

7s:

Insect pests of maize
and
their management

ICAR - Indian Institute of Maize Research
Ludhiana-141 004, Punjab

Maize (Zea mays L.) is the third most important cereal crop after rice and wheat being
grow

Result 3
Source: cabbageegnlish_ (1).pdf
Chunk: 0
Distance: 0.3353
Text: C
ole crops are attacked by a number of important insect
pests and diseases like diamond back moth, Plutella
xylostella; tobacco 

## 9. Inspect retrieval quality

The query returns the five closest chunks according to cosine distance. **Lower distance indicates greater similarity.** Inspect the returned text and source metadata to judge whether the results answer the query; semantic similarity does not guarantee that every retrieved chunk is relevant.

Rank | Source | Distance | Relevance

1	| Maize | insect pests | 0.3082 | Excellent

2 | Maize | insect pests | 0.3160 | Excellent

3 | Cabbage | pests | 0.3353 | Less relevant

4 | Maize | insect pests | 0.3839 | Relevant

5 | Vegetable | crops | 0.4082 | Partially relevant

## 10. Generate a grounded answer with Gemini

The local embedding model is used for retrieval, while Gemini generates the natural-language answer. Source references are printed directly from ChromaDB metadata rather than asking the language model to invent citation details.

In [ ]:
api_key = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

MODEL = "gemini-3.5-flash-lite"

print("Gemini client initialized successfully!")

In [ ]:
query = "What are the major insect pests affecting maize in India?"

query_embedding = embedding_model.encode(
    [query],
    normalize_embeddings=True
)[0]

query_results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=5
)

context = "\n\n".join(query_results["documents"][0])

prompt = f"""
You are an agriculture knowledge assistant.

Answer the question using ONLY the provided context.
If the answer is unavailable, reply exactly:
I don't know from these documents.

Do not generate source citations yourself.

Context:
{context}

Question:
{query}
"""

response = client.models.generate_content(
    model=MODEL,
    contents=prompt
)

print("Answer:\n")
print(response.text)

print("\nSources:")
for metadata in query_results["metadatas"][0]:
    print(f"- {metadata['filename']} — Chunk {metadata['chunk_number']}")

## 11. Test refusal for unsupported questions

Ask a question that is outside the agriculture documents. The expected response is exactly:

`I don't know from these documents.`

In [ ]:
unsupported_query = "Who invented the telephone?"

unsupported_embedding = embedding_model.encode(
    [unsupported_query],
    normalize_embeddings=True
)[0]

unsupported_results = collection.query(
    query_embeddings=[unsupported_embedding.tolist()],
    n_results=5
)

unsupported_context = "\n\n".join(
    unsupported_results["documents"][0]
)

refusal_prompt = f"""
You are an agriculture knowledge assistant.

Answer using ONLY the provided context.
If the answer is not available in the context, reply exactly:
I don't know from these documents.

Context:
{unsupported_context}

Question:
{unsupported_query}
"""

refusal_response = client.models.generate_content(
    model=MODEL,
    contents=refusal_prompt
)

print(refusal_response.text)

## 12. Conclusion

The agriculture RAG pipeline was implemented and tested.

- Collected and processed 10 agriculture PDF documents.
- Used native PDF text extraction and OCR for scanned maize documents.
- Created 1,478 chunks using 500-character chunking with 50-character overlap.
- Generated 1,478 local embeddings using `all-MiniLM-L6-v2` (384 dimensions).
- Stored vectors, chunk text, and source metadata in ChromaDB.
- Retrieved the top 5 chunks for an agriculture query.
- Generated an answer grounded in retrieved context and displayed source references.
- Tested the required refusal response for an unsupported question.

**Observation:** Retrieval returned highly relevant maize-pest chunks at the top, but some unrelated crop-pest chunks also appeared. This indicates that retrieval quality can be improved further through better chunking, filtering, or reranking.